# ENGR 1451/2451 — Weekly Assignment 10
**10 points**

This assignment focuses on three goals:
(i) interpreting Type I and Type II errors, sensitivity, and positive predictive value using **real biopsy-image data**;
(ii) investigating how significance cutoffs and sample size affect power, using measured features from the same dataset; and
(iii) comparing independent groups and paired observations with $t$-tests.


**Due:** See Canvas.

Complete the work directly in this notebook. Show your Python code and give brief written explanations where requested. Submit the completed notebook as directed in Canvas.

## What to hand in

Work this assignment with the **course tutor** in the class project. That is the AI help to use on this assignment, and it keeps the record for you — you do not write one yourself.

At the end of each working session, type **"make my record"** in the chat. Copy the block it produces and paste it into the **Session records** cell near the bottom of this notebook.

You do not have to finish in one sitting. To pick up later, start a new chat in the same project, say which assignment you are on and which problem you stopped at, and keep going. That session gives you its own block, which you paste under the previous one.

Records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. The code and the written answers still have to be yours.

## LE10-1. Real data: diagnostic flags and base rates (3 pts)

**Dataset:** `./data/LE10_wisconsin_tumors.csv` contains **569 real, labeled biopsy-image observations** from the Wisconsin Diagnostic Breast Cancer dataset. Each row represents a breast mass, with a known diagnosis (`benign` or `malignant`) and measurements extracted from images of cell nuclei. You will use the feature `mean_radius`.

For this classroom exercise, suppose a simple imaging rule **flags** a case when `mean_radius >= 14.0`. A flag means **the rule predicts malignant**; a non-flag means it predicts benign. The threshold is chosen for teaching, **not** as a validated medical screening rule.

The data come from a selected group of diagnostic cases, **not a random population-screening sample**. We will therefore distinguish (1) rates calculated from these real observations from (2) projections under a *hypothetical* population prevalence.

**Source:** [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast-cancer-wisconsin-diagnostic), Wolberg, Mangasarian, Street & Street (1993), DOI: 10.24432/C5DW2B (CC BY 4.0). This teaching CSV contains original real observations but only two of the measured features.

### A. Build the screening table and examine prevalence (2 pts)

Using Python:

1. Load the CSV with `pd.read_csv()` and show the number of cases in each diagnosis category.
2. Using the fixed threshold above, count **true positives (TP)**, **false negatives (FN)**, **false positives (FP)**, and **true negatives (TN)**. Treat *malignant* as the positive class.
3. Display a $2\times 2$ `pd.DataFrame` with **Flagged / Not flagged** as rows and **Malignant / Benign** as columns.
4. Calculate the observed **sensitivity**, **specificity**, **positive predictive value (PPV)**, and **accuracy**:

$$
\mathrm{Sensitivity}=\frac{TP}{TP+FN},\quad
\mathrm{Specificity}=\frac{TN}{TN+FP},\quad
\mathrm{PPV}=\frac{TP}{TP+FP},\quad
\mathrm{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}.
$$

5. **Base-rate thought experiment:** Suppose this rule is applied to **200,000** future cases in a hypothetical population where **0.5%** are malignant. Assume the sensitivity and specificity found in this CSV would transfer unchanged. Compute the *expected* TP, FN, FP, and TN, then the projected PPV. Expected counts need not be whole numbers; round only when displaying them. Compare the projected PPV with the PPV from the observed dataset.

Report percentages to **one decimal place**.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.precision", 4)

cases = pd.read_csv("./data/LE10_wisconsin_tumors.csv")
display(cases.head())
print(cases["diagnosis"].value_counts())

threshold = 14.0
# A case is flagged when mean_radius >= threshold.
# Calculate TP, FN, FP, TN, then display the 2 x 2 screening table.
# Compute empirical sensitivity, specificity, PPV, and accuracy.
# Project expected screening outcomes for 200,000 cases with 0.5% malignancy.
# Compute the projected PPV (using unrounded expected counts).

# Your code here


### B. Interpret the results (1 pt)

Answer in about **3–4 sentences**:

1. For this screening rule, what is a **Type I error** (false positive)? What is a **Type II error** (false negative)?
2. Why does the PPV change so much when the assumed prevalence changes, even if sensitivity and specificity stay fixed? Does a specificity near 85% imply that 85% of flagged cases are malignant?
3. Why should you **not** claim that the observed proportion of malignant cases in this dataset is the prevalence in the general population?

**Answer:**


## LE10-2. Significance cutoffs, $\beta$, and power (3 pts)

Use the **same real CSV**, but now study another imaging feature, `mean_smoothness`. We are designing a **population-level test based on the mean of $n$ independent biopsies**, *not* screening an individual case. Unlike LE10-1, which uses a fixed threshold to classify individual cases, we now choose cutoffs to control the probability of a Type I error at a specified significance level, $\alpha$.

Estimate the mean and sample SD of `mean_smoothness` separately for **benign** and **malignant** cases in the CSV. For this *planning exercise*, treat those estimated values as fixed benchmarks. Under a **normal approximation for sample means**, model

$$
\bar X\mid H_0 \approx N\!\left(\mu_0,\frac{\sigma_0^2}{n}\right),\qquad
\bar X\mid H_A \approx N\!\left(\mu_1,\frac{\sigma_1^2}{n}\right),
$$

where $H_0$ uses the **benign** reference distribution and the specified $H_A$ uses the **malignant** distribution. A false alarm means rejecting the benign reference when it is true; a miss means failing to reject it when the malignant model is true. **These are approximate design calculations**, not an evaluation of a clinical test or a claim that all biopsy measurements are normally distributed.

Use a **two-sided** decision rule: flag a sample if its mean is below the lower cutoff **or** above the upper cutoff. Compute cutoffs from the *benign* sampling distribution; compute $\beta$ as the probability of falling **between those cutoffs under the malignant model**.

| Plan | Independent biopsies $n$ | Significance level $\alpha$ |
|---|---:|---:|
| A | 10 | 0.05 |
| B | 10 | 0.01 |
| C | 25 | 0.01 |

### A. Calculate error rates and power (2 pts)

Using Python:

1. Extract `mean_smoothness` for benign and malignant cases; calculate each group's sample mean and sample SD using `.std(ddof=1)`.
2. For each plan, define `benign_model` and `malignant_model` with `stats.norm(loc=..., scale=SD/np.sqrt(n))`.
3. Calculate the **lower and upper cutoffs** using `benign_model.ppf([alpha/2, 1-alpha/2])`.
4. Compute $\beta= P(\text{not flagged}\mid H_A)$ using `malignant_model.cdf(upper) - malignant_model.cdf(lower)` and **power** $=1-\beta$.
5. Display a table with plan, $n$, $\alpha$, both cutoffs, $\beta$, and power. Report probabilities to **three decimal places**.

*Use the actual measurements to estimate model parameters; do not simulate additional biopsy records.*


In [ ]:
# Use the dataset loaded in LE10-1.
benign = cases.loc[cases["diagnosis"] == "benign", "mean_smoothness"]
malignant = cases.loc[cases["diagnosis"] == "malignant", "mean_smoothness"]

# Calculate mu0, sd0 (benign), and mu1, sd1 (malignant) from the CSV.
# Print both pairs of estimates.

plans = pd.DataFrame({
    "plan": ["A", "B", "C"],
    "n": [10, 10, 25],
    "alpha": [0.05, 0.01, 0.01],
})

# For each plan:
#   - Define benign_model and malignant_model using stats.norm(...).
#   - Calculate cutoffs using benign_model.ppf(...).
#   - Calculate beta using malignant_model.cdf(...).
#   - Calculate power = 1 - beta.
# Display a labeled results table.

# Your code here


### B. Explain the tradeoffs (1 pt)

Answer in about **2–3 sentences**:

1. Compare plans **A and B**: What happens to false alarms ($\alpha$) and missed detections ($\beta$) when the cutoff becomes stricter but $n$ stays the same?
2. Compare plans **B and C**: How does using more **independent biopsies** increase power at the same $\alpha$? Why would this be a study-design consideration rather than a way to improve diagnosis of a *single* patient?

**Answer:**


## LE10-3. Comparing two mean temperatures (4 pts)

Choose the analysis based on the **study design**, as in w10b. Both parts concern peak battery-module temperature during a standardized load test; **lower is better**.

*The datasets in this problem are simulated classroom examples, not measurements from a published study.*

### A. Independent modules: new vs. standard cooling (2.5 pts)

The file `./data/LE10_battery_cooling.csv` contains peak temperatures ($^\circ$C) for **26 different battery modules**: 12 were randomly assigned to **new** cooling and 14 to **standard** cooling. Each module was measured once.

Let $\Delta=\mu_{\mathrm{new}}-\mu_{\mathrm{standard}}$. We want to know whether the new method **reduces** the population mean peak temperature. Test the boundary null $H_0:\Delta=0$ against $H_A:\Delta<0$, using $\alpha=0.05$.

Assume observations from different modules are independent, group distributions are approximately normal without strong outliers, and population SDs are unknown. Use **Welch's** two-sample $t$-test (do not assume equal population variances).

In [ ]:
cooling = pd.read_csv("./data/LE10_battery_cooling.csv")

# The order is new minus standard: a negative difference favors the new method.
new = cooling.loc[cooling["method"] == "new", "peak_temperature_C"].to_numpy()
standard = cooling.loc[cooling["method"] == "standard", "peak_temperature_C"].to_numpy()

display(cooling.head())
print(cooling["method"].value_counts())

Use Python to:

1. Calculate each group's **sample size, mean, and sample SD** (with `ddof=1`), and the observed difference $\bar x_{\mathrm{new}}-\bar x_{\mathrm{standard}}$.
2. Calculate the estimated SE of the difference:

$$
\widehat{SE}=\sqrt{\frac{s_{\mathrm{new}}^2}{n_{\mathrm{new}}}+\frac{s_{\mathrm{standard}}^2}{n_{\mathrm{standard}}}}.
$$

3. Run `stats.ttest_ind(new, standard, equal_var=False, alternative="less")`. Print the **$t$ statistic, one-sided $P$-value**, and decision at $\alpha=0.05$.
4. For context, run the same Welch test with `alternative="two-sided"` and use its `.confidence_interval(confidence_level=0.95)` method to report a **two-sided 95% confidence interval** for $\Delta$. (You do not need to calculate Welch degrees of freedom manually.)

**Answer:**

In [ ]:
# Calculate group summaries (n, sample mean, sample SD).
# Calculate the observed difference and estimated SE.
# Run Welch's one-sided test and print the decision at alpha = 0.05.
# Run a two-sided Welch test and print its 95% confidence interval.

# Your code here

### B. Matched modules: before vs. after a firmware change (1.5 pts)

A **separate experiment** measures the *same eight modules* before and after a cooling-control firmware update. The arrays are aligned: `before[i]` and `after[i]` belong to the same module.

1. Calculate the eight within-module differences `after - before` and their mean.
2. Run `stats.ttest_rel(after, before, alternative="less")` to test $H_0:\mu_d=0$ against $H_A:\mu_d<0$, where $d=\text{after}-\text{before}$. Print $t$, the one-sided $P$-value, and the decision at $\alpha=0.05$.
3. In **2–3 sentences total**, interpret the conclusions in parts A and B. Explain **why a paired rather than independent test** is required in part B and whether a statistically significant result by itself establishes the engineering importance of a temperature change.

**Answer:**

In [ ]:
# Same modules measured twice; each index identifies the same module.
before = np.array([72, 68, 71, 70, 74, 69, 73, 67])
after  = np.array([69, 65, 70, 67, 73, 66, 72, 66])

# Calculate and display within-module differences and their mean.
# Run the paired t-test, report t and P, and make the decision at alpha = 0.05.

# Your code here

## Session records

At the end of each working session, type **"make my record"** in the chat and paste the block it produces into the cell below. If you work across several sessions, paste each new block underneath the previous one.

Session records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. They are read to find out which steps are hardest, and used to improve the course.

<!-- Paste your session record blocks in this cell, one after another. -->

*(session records go here)*

## Before you submit

1. Keep `LE10.ipynb` next to a `data` folder containing **both** `LE10_wisconsin_tumors.csv` and the **original** `LE10_battery_cooling.csv`. This revision provides the real Wisconsin CSV; the battery-cooling CSV from the earlier version of LE10 is still needed for Question 3.
2. **Restart Kernel and Run All Cells** after both CSV files are present and confirm every completed cell runs without errors.
3. Confirm your calculated tables, printed values, and written answers are visible in the saved notebook.
4. Confirm your session record blocks are pasted into the cell above.
5. Save the notebook, then upload the `.ipynb` to Canvas, as directed.
